# Sprint 10 - Segmentation + Backbone-lr Sweep

**Two experiments, one notebook:**

**Experiment 1 (Steps 5-9): Segmentation diagnostic** — strip backgrounds from PlantDoc
test images with GrabCut, eval existing model. Fast (~1 hour, CPU only).
Tells us if background removal helps before committing GPU time.

**Experiment 2 (Steps 10-12): Backbone-lr + mixed-ratio sweep** — the highest-leverage
experiment. Combines two proven ingredients for the first time:
- Unfrozen backbone at lr=1e-5 (gave 0.66 field on PD-only)
- Mixed PV+PD training (gave 0.95+ lab retention)
Sweep plantdoc-repeat: 15, 20, 30. If segmentation helped, run on segmented data.

**Order:** segmentation first (fast, informs sweep data choice), then sweep.

**Gates:** field F1 >= 0.60 AND lab F1 >= 0.60
**If both pass:** take that checkpoint, go to frontend.
**If not:** report results, decide next steps.

In [ ]:
import os
import platform
import subprocess
import sys

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
try:
    gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30)
    print(gpu.stdout.strip().splitlines()[0] if gpu.stdout.strip() else gpu.stderr.strip() or "No GPU detected (CPU only)")
except Exception as exc:
    print("GPU check skipped:", exc)

## Step 1 - Mount Drive + clone repo + install deps

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

REPO_URL = "https://github.com/io-PEAK/folium.git"
REPO_DIR = Path("/content/folium")
DATA_DIR = Path("/content/drive/MyDrive/folium/data")
LOCAL_RAW_DIR = Path("/content/folium_raw")
LOCAL_DATA_DIR = Path("/content/folium_data")
CHECKPOINT_DIR = Path("/content/drive/MyDrive/folium/checkpoints")
RESULTS_DIR = Path("/content/drive/MyDrive/folium/results")

if not (REPO_DIR / "ml").exists():
    %cd /content
    !git clone --depth 1 {REPO_URL}
else:
    !git -C {REPO_DIR} pull --ff-only -q

for d in (DATA_DIR, LOCAL_RAW_DIR, LOCAL_DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("DATA_DIR (archives):", DATA_DIR)
print("LOCAL_DATA_DIR:", LOCAL_DATA_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

def run(cmd, cwd, label, stream=False):
    env = dict(os.environ, PYTHONPATH=str(REPO_DIR))
    if stream:
        proc = subprocess.Popen(
            cmd, cwd=str(cwd), env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        lines = []
        for line in proc.stdout:
            print(line, end="")
            lines.append(line)
        proc.wait()
        combined = "".join(lines)
        if proc.returncode != 0:
            print(f"\n[{label}] failed (returncode {proc.returncode})")
        assert proc.returncode == 0, label
        class _Result:
            pass
        r = _Result()
        r.stdout = combined
        r.stderr = ""
        r.returncode = 0
        return r
    result = subprocess.run(cmd, cwd=str(cwd), env=env, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"[{label}] failed (returncode {result.returncode})")
        print("stdout tail:\n", result.stdout[-2000:])
        print("stderr tail:\n", result.stderr[-2000:])
    assert result.returncode == 0, label
    return result

In [ ]:
%pip install -q torch torchvision albumentations matplotlib pandas tqdm opencv-python-headless scikit-learn

## Step 2 - Clean old Sprint 9 rows from ablation CSV

In [ ]:
import pandas as pd

csv_path = RESULTS_DIR / "ablation_results.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
    old = df["variant"].str.startswith("s9_") | df["variant"].str.startswith("s10_")
    n_old = old.sum()
    if n_old > 0:
        df = df[~old].reset_index(drop=True)
        df.to_csv(csv_path, index=False)
        print(f"Removed {n_old} old s9/s10 rows from {csv_path}")
    else:
        print("No old s9/s10 rows found.")
else:
    print("No ablation CSV yet.")

## Step 3 - Hydrate raw from Drive, then organize splits locally

In [ ]:
import sys

sys.path.insert(0, str(REPO_DIR))
from scripts.download_datasets import PLANTVILLAGE_EXPECTED, PLANTDOC_EXPECTED, hydrate_dataset

for name, expected in (("plantvillage", PLANTVILLAGE_EXPECTED), ("plantdoc", PLANTDOC_EXPECTED)):
    try:
        hydrate_dataset(LOCAL_RAW_DIR, DATA_DIR, name, expected)
    except RuntimeError as exc:
        print("HYDRATE FAILED:", exc)
        raise

result = run([
    sys.executable,
    str(REPO_DIR / "scripts" / "organize_datasets.py"),
    "--raw-dir", str(LOCAL_RAW_DIR),
    "--data-dir", str(LOCAL_DATA_DIR),
    "--seed", "42",
    "--val-fraction", "0.15",
    "--test-fraction", "0.15",
], cwd=str(REPO_DIR), label="organize_datasets.py failed")
print("Data ready at", LOCAL_DATA_DIR)

## Step 4 - List available checkpoints

We need the `mixed` checkpoint (v13: fine-tune-then-mix, single-head trained on PV+PD combined).
Also list the dual-head domain checkpoint for comparison.

In [ ]:
ckpts = sorted(CHECKPOINT_DIR.glob("best_*.pt"))
print(f"Found {len(ckpts)} checkpoints:")
for c in ckpts:
    size_mb = c.stat().st_size / 1024 / 1024
    print(f"  {c.name} ({size_mb:.0f}MB)")

## Step 5 - Fix mislabeled PlantDoc test images

The audit (Sprint 9) found ~10 images where the model is >80% confident
and the label is likely wrong (same-species disease confusion).
We re-run the audit, then move those images to the correct class folders.

This improves measured F1 because the model was right all along.

In [ ]:
AUDIT_CSV = RESULTS_DIR / "plantdoc_label_audit_s10.csv"

# Find the domain checkpoint for the audit
domain_candidates = list(CHECKPOINT_DIR.glob("best_domain_*.pt"))
if domain_candidates:
    DOMAIN_CKPT = domain_candidates[-1]  # most recent
    print(f"Using domain checkpoint for audit: {DOMAIN_CKPT.name}")

    cmd = [
        sys.executable, str(REPO_DIR / "scripts" / "audit_plantdoc_labels.py"),
        "--checkpoint", str(DOMAIN_CKPT),
        "--data-dir", str(LOCAL_DATA_DIR),
        "--split", "test",
        "--dual-head",
        "--separate-backbones",
        "--out", str(AUDIT_CSV),
    ]
    result = run(cmd, cwd=str(REPO_DIR), label="audit failed", stream=True)
else:
    print("WARNING: No domain checkpoint found. Run Sprint 9 first.")
    AUDIT_CSV = None

In [ ]:
if AUDIT_CSV and AUDIT_CSV.exists():
    # First: dry run to see what would be fixed
    cmd = [
        sys.executable, str(REPO_DIR / "scripts" / "fix_plantdoc_labels.py"),
        "--audit-csv", str(AUDIT_CSV),
        "--data-dir", str(LOCAL_DATA_DIR),
        "--dataset", "plantdoc",
        "--split", "test",
        "--dry-run",
    ]
    result = run(cmd, cwd=str(REPO_DIR), label="fix audit failed", stream=True)

Review the dry-run output above. If the fixes look correct, run the next cell
to actually move the files. Remove `--dry-run` to apply.

**⚠ Circular reasoning warning:** this uses the model's own predictions to correct
ground truth labels that will then grade descendant models. Only fix images you
personally verify by looking at them — don't trust the audit script blindly.

In [ ]:
# Uncomment the line below to actually move files (remove --dry-run)
# cmd = [
#     sys.executable, str(REPO_DIR / "scripts" / "fix_plantdoc_labels.py"),
#     "--audit-csv", str(AUDIT_CSV),
#     "--data-dir", str(LOCAL_DATA_DIR),
#     "--dataset", "plantdoc",
#     "--split", "test",
#     "--log", str(RESULTS_DIR / "s10_label_fixes.csv"),
# ]
# result = run(cmd, cwd=str(REPO_DIR), label="fix failed", stream=True)

## Step 6 - Segment PlantDoc test images

Run OpenCV's GrabCut on every PlantDoc test image to isolate the leaf.
Saves cropped leaf images to `/tmp/seg_plantdoc_test/<class>/image.jpg`.

In [ ]:
from PIL import Image
import cv2
import numpy as np
from tqdm import tqdm

src_dir = LOCAL_DATA_DIR / "plantdoc" / "test"
dst_dir = Path("/tmp/seg_plantdoc_test")

if dst_dir.exists():
    import shutil
    shutil.rmtree(dst_dir)

def segment_leaf(img_bgr, max_dim=800):
    """Segment leaf from background using GrabCut.
    Returns RGB image with background replaced by white (matching PV segmented convention).
    Downscales large images first (GrabCut is slow/can hang on big images), then
    rescales the crop coordinates back up so output stays close to original resolution.
    """
    orig_h, orig_w = img_bgr.shape[:2]
    scale = min(1.0, max_dim / max(orig_h, orig_w))
    if scale < 1.0:
        small = cv2.resize(img_bgr, (int(orig_w * scale), int(orig_h * scale)))
    else:
        small = img_bgr
    h, w = small.shape[:2]
    mask = np.zeros((h, w), np.uint8)
    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)
    rect = (5, 5, w - 10, h - 10)
    cv2.grabCut(small, mask, rect, bgd_model, fgd_model, 3, cv2.GC_INIT_WITH_RECT)
    # Binary mask: definite/possible foreground = 255, rest = 0
    mask2 = np.where((mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD), 255, 0).astype(np.uint8)
    # Mask out background: set to white (PV segmented convention)
    result = np.full_like(small, 255)
    result[mask2 == 255] = small[mask2 == 255]
    # Crop to bounding box of the leaf
    coords = cv2.findNonZero(mask2)
    if coords is None:
        return cv2.cvtColor(small, cv2.COLOR_BGR2RGB)
    x, y, bw, bh = cv2.boundingRect(coords)
    pad = 10
    x1, y1 = max(0, x - pad), max(0, y - pad)
    x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
    cropped = result[y1:y2, x1:x2]
    return cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)

# Collect all images
all_images = []
for class_dir in sorted(src_dir.iterdir()):
    if not class_dir.is_dir():
        continue
    for img_path in sorted(class_dir.glob("*")):
        if img_path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}:
            all_images.append((class_dir.name, img_path))

print(f"Segmenting {len(all_images)} images with GrabCut...")
count = 0
errors = 0
for class_name, img_path in tqdm(all_images, desc="Segmenting"):
    dst_class = dst_dir / class_name
    dst_class.mkdir(parents=True, exist_ok=True)
    try:
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            raise ValueError("cv2.imread returned None")
        result_rgb = segment_leaf(img_bgr)
        Image.fromarray(result_rgb).save(dst_class / img_path.name)
        count += 1
    except Exception as e:
        errors += 1
        print(f"  ERROR on {img_path.name}: {e}")

print(f"Done: {count} segmented, {errors} errors")

## Step 7 - Visual spot-check

Render 20 segmented images side-by-side with originals.
Check that the segmenter is removing background, not cutting into leaves.

In [ ]:
import matplotlib.pyplot as plt

src_images = sorted([p for p in src_dir.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])[:20]

n = len(src_images)
cols = 5
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols * 2, figsize=(24, 4 * rows))
if rows == 1:
    axes = [axes]

for i, src_img in enumerate(src_images):
    r, c = divmod(i, cols)
    orig = Image.open(src_img).convert("RGB")
    seg_path = dst_dir / src_img.parent.name / src_img.name
    seg = Image.open(seg_path).convert("RGB") if seg_path.exists() else orig

    axes[r][c * 2].imshow(orig)
    axes[r][c * 2].set_title("original", fontsize=8)
    axes[r][c * 2].axis("off")

    axes[r][c * 2 + 1].imshow(seg)
    axes[r][c * 2 + 1].set_title("segmented", fontsize=8)
    axes[r][c * 2 + 1].axis("off")

# hide unused axes
for i in range(n, rows * cols):
    r, c = divmod(i, cols)
    axes[r][c * 2].axis("off")
    axes[r][c * 2 + 1].axis("off")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "s10_spot_check.png", dpi=120)
print("Saved", RESULTS_DIR / "s10_spot_check.png")
plt.show()

## Step 8 - Evaluate on segmented PlantDoc test

Run existing checkpoints on the segmented test images.
We evaluate:
1. The `mixed` checkpoint (v13, single-head, trained on PV+PD) — our best all-rounder
2. The domain-routed dual-head checkpoint (Sprint 9) — for comparison

We create a temporary data directory with the segmented images + class_map.json,
then point `ml/evaluate.py` at it.

In [ ]:
import json
import shutil

# Create temporary data directory with segmented PD test
seg_data_dir = Path("/tmp/seg_data")
if seg_data_dir.exists():
    shutil.rmtree(seg_data_dir)
seg_data_dir.mkdir()

# Symlink plantdoc/test to segmented images
(seg_data_dir / "plantdoc").mkdir()
(seg_data_dir / "plantdoc" / "test").symlink_to(dst_dir.resolve())

# Copy class_map.json (needed for --map-to-pv)
shutil.copy(LOCAL_DATA_DIR / "class_map.json", seg_data_dir / "class_map.json")

# Also need plantvillage test dir (evaluate may reference it)
pv_test = LOCAL_DATA_DIR / "plantvillage" / "test"
if pv_test.exists():
    (seg_data_dir / "plantvillage").mkdir(exist_ok=True)
    (seg_data_dir / "plantvillage" / "test").symlink_to(pv_test.resolve())

print("Segmented data dir:", seg_data_dir)
if dst_dir.exists():
    print("PD test classes:", sorted([d.name for d in (seg_data_dir / "plantdoc" / "test").iterdir()]))
else:
    print("WARNING: Segmented images not found. Run Step 5 first.")

In [ ]:
# Find the mixed checkpoint (v13)
mixed_candidates = list(CHECKPOINT_DIR.glob("best_plantvillage_mixed*.pt"))
if mixed_candidates:
    MIXED_CKPT = mixed_candidates[0]
    print(f"Using mixed checkpoint: {MIXED_CKPT.name}")
else:
    print("WARNING: No mixed checkpoint found. Available checkpoints:")
    for c in sorted(CHECKPOINT_DIR.glob("best_*.pt")):
        print(f"  {c.name}")
    print("\nSet MIXED_CKPT manually below.")
    MIXED_CKPT = None

In [ ]:
# 1) Mixed checkpoint on segmented PD test (single-head, no --dual-head)
if MIXED_CKPT:
    print("=" * 60)
    print("8a) mixed on SEGMENTED PlantDoc test")
    print("=" * 60)
    cmd = [
        sys.executable, "-m", "ml.evaluate",
        "--checkpoint", str(MIXED_CKPT),
        "--data-dir", str(seg_data_dir),
        "--dataset", "plantdoc",
        "--split", "test",
        "--map-to-pv",
        "--variant", "s10_seg_mixed_field",
        "--results", str(RESULTS_DIR / "ablation_results.csv"),
    ]
    result = run(cmd, cwd=str(REPO_DIR), label="eval failed", stream=True)

    # 2) Same checkpoint on original PD test (baseline comparison)
    print("\n" + "=" * 60)
    print("8b) mixed on ORIGINAL PlantDoc test (baseline)")
    print("=" * 60)
    cmd = [
        sys.executable, "-m", "ml.evaluate",
        "--checkpoint", str(MIXED_CKPT),
        "--data-dir", str(LOCAL_DATA_DIR),
        "--dataset", "plantdoc",
        "--split", "test",
        "--map-to-pv",
        "--variant", "s10_orig_mixed_field",
        "--results", str(RESULTS_DIR / "ablation_results.csv"),
    ]
    result = run(cmd, cwd=str(REPO_DIR), label="eval failed", stream=True)

    # 3) Sanity check: same checkpoint on PV test (should not drop)
    print("\n" + "=" * 60)
    print("8c) mixed on PlantVillage test (sanity check, should stay ~0.95)")
    print("=" * 60)
    cmd = [
        sys.executable, "-m", "ml.evaluate",
        "--checkpoint", str(MIXED_CKPT),
        "--data-dir", str(LOCAL_DATA_DIR),
        "--dataset", "plantvillage",
        "--split", "test",
        "--variant", "s10_orig_mixed_lab",
        "--results", str(RESULTS_DIR / "ablation_results.csv"),
    ]
    result = run(cmd, cwd=str(REPO_DIR), label="eval failed", stream=True)

## Step 9 - Verdict

Compare field F1 on segmented vs original PlantDoc test.

**If segmented F1 >= 0.50:** segmentation helps -> proceed to Sprint 11 (retrain on segmented data)

**If segmented F1 < 0.50:** background isn't the main issue -> fall back to Plan A (repeat sweep)

In [ ]:
df = pd.read_csv(str(RESULTS_DIR / "ablation_results.csv")).drop_duplicates()

seg_key = "s10_seg_mixed_field"
orig_key = "s10_orig_mixed_field"
lab_key = "s10_orig_mixed_lab"

seg_row = df[df["variant"] == seg_key]
orig_row = df[df["variant"] == orig_key]
lab_row = df[df["variant"] == lab_key]

print("=== Segmentation effect on field F1 ===")
if len(orig_row) > 0 and len(seg_row) > 0:
    orig_f1 = orig_row["f1"].iloc[0]
    seg_f1 = seg_row["f1"].iloc[0]
    delta = seg_f1 - orig_f1
    print(f"Original PD test:  {orig_f1:.4f}")
    print(f"Segmented PD test: {seg_f1:.4f}")
    print(f"Delta:             {delta:+.4f}")
    if seg_f1 >= 0.50:
        print(f"\nVerdict: PASS ({seg_f1:.4f} >= 0.50) -> segmentation helps, proceed to retraining")
    else:
        print(f"\nVerdict: FAIL ({seg_f1:.4f} < 0.50) -> background not the main bottleneck, try Plan A")
else:
    print("Missing results. Run Step 7 first.")

if len(lab_row) > 0:
    print(f"\nLab sanity check: {lab_row['f1'].iloc[0]:.4f} (should be ~0.95)")
    if lab_row["f1"].iloc[0] < 0.85:
        print("WARNING: lab dropped! Segmenter may be cutting into leaves.")

---
## Step 9b - PV segmented baseline

Before deciding on `USE_SEGMENTED`, eval the `mixed` checkpoint on PlantVillage's
own `raw/segmented` test set. This tells you: does the model already handle
flat-background photos? If PV-segmented score drops vs original PV, the model
isn't used to segmented images and a flat-background PD score is meaningless.

In [ ]:
# Check if PV segmented data exists on Drive
pv_seg_raw = DATA_DIR / "plantvillage" / "raw" / "segmented"
if not pv_seg_raw.exists():
    # Try alternative paths
    for alt in [DATA_DIR / "raw" / "segmented", Path("/content/folium_raw") / "plantvillage" / "segmented"]:
        if alt.exists():
            pv_seg_raw = alt
            break

if pv_seg_raw.exists() and MIXED_CKPT:
    print(f"PV segmented data found at: {pv_seg_raw}")
    print("Contents:", sorted([d.name for d in pv_seg_raw.iterdir()]))

    # Build a temp data dir with PV segmented test
    pv_seg_eval = Path("/tmp/pv_seg_eval")
    if pv_seg_eval.exists():
        import shutil
        shutil.rmtree(pv_seg_eval)
    pv_seg_eval.mkdir()

    # Check if segmented has train/val/test splits or just images
    has_splits = any((pv_seg_raw / s).exists() for s in ["train", "val", "test"])
    if has_splits:
        (pv_seg_eval / "plantvillage").mkdir()
        for split in ["train", "val", "test"]:
            split_dir = pv_seg_raw / split
            if split_dir.exists():
                (pv_seg_eval / "plantvillage" / split).symlink_to(split_dir.resolve())
        # Also symlink class_map.json
        import shutil
        shutil.copy(LOCAL_DATA_DIR / "class_map.json", pv_seg_eval / "class_map.json")
    else:
        print("WARNING: PV segmented has no train/val/test splits. Cannot eval.")
        pv_seg_eval = None

    if pv_seg_eval:
        print("=" * 60)
        print("9b) mixed on PV SEGMENTED test (should match original ~0.95)")
        print("=" * 60)
        cmd = [
            sys.executable, "-m", "ml.evaluate",
            "--checkpoint", str(MIXED_CKPT),
            "--data-dir", str(pv_seg_eval),
            "--dataset", "plantvillage",
            "--split", "test",
            "--variant", "s10_pv_seg_baseline",
            "--results", str(RESULTS_DIR / "ablation_results.csv"),
        ]
        result = run(cmd, cwd=str(REPO_DIR), label="eval PV segmented failed", stream=True)

        # Compare with original PV test
        df = pd.read_csv(str(RESULTS_DIR / "ablation_results.csv")).drop_duplicates()
        seg_row = df[df["variant"] == "s10_pv_seg_baseline"]
        orig_row = df[df["variant"] == "s10_orig_mixed_lab"]
        if len(seg_row) > 0 and len(orig_row) > 0:
            print(f"\nOriginal PV test:  {orig_row['f1'].iloc[0]:.4f}")
            print(f"Segmented PV test: {seg_row['f1'].iloc[0]:.4f}")
            delta = seg_row['f1'].iloc[0] - orig_row['f1'].iloc[0]
            print(f"Delta:             {delta:+.4f}")
            if abs(delta) < 0.03:
                print("Model handles segmented images fine. Proceed with USE_SEGMENTED decision.")
            else:
                print(f"WARNING: PV score {'dropped' if delta < 0 else 'changed'} on segmented data.")
                print("A flat PD field score may reflect domain mismatch, not background removal benefit.")
else:
    print("PV segmented data not found on Drive. Skipping baseline.")
    print("Checked:", pv_seg_raw)

---
## Step 6b - Segment PD train+val (only if Step 9 showed segmentation helps)

If you set `USE_SEGMENTED = True` below, the sweep needs segmented train/val data
too. This step segments PD train+val images (~1300 images, ~30-40 min on CPU).
If `USE_SEGMENTED = False`, skip this cell.

In [ ]:
# Only run this if you decided to use segmented data
if not USE_SEGMENTED:
    print("USE_SEGMENTED=False, skipping PD train+val segmentation.")
else:
    print("Segmenting PD train+val images...")
    for split in ["train", "val"]:
        src_split = LOCAL_DATA_DIR / "plantdoc" / split
        dst_split = Path(f"/tmp/seg_plantdoc_{split}")
        if dst_split.exists():
            import shutil
            shutil.rmtree(dst_split)

        all_images = []
        for class_dir in sorted(src_split.iterdir()):
            if not class_dir.is_dir():
                continue
            for img_path in sorted(class_dir.glob("*")):
                if img_path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}:
                    all_images.append((class_dir.name, img_path))

        print(f"  {split}: {len(all_images)} images")
        count, errors = 0, 0
        for class_name, img_path in tqdm(all_images, desc=f"Segmenting {split}"):
            dst_class = dst_split / class_name
            dst_class.mkdir(parents=True, exist_ok=True)
            try:
                img_bgr = cv2.imread(str(img_path))
                if img_bgr is None:
                    raise ValueError("cv2.imread returned None")
                result_rgb = segment_leaf(img_bgr)
                Image.fromarray(result_rgb).save(dst_class / img_path.name)
                count += 1
            except Exception as e:
                errors += 1
                print(f"    ERROR on {img_path.name}: {e}")
        print(f"  {split}: {count} done, {errors} errors")

    # Build full segmented data dir for sweep
    seg_data_dir = Path("/tmp/seg_data")
    if seg_data_dir.exists():
        import shutil
        shutil.rmtree(seg_data_dir)
    seg_data_dir.mkdir()

    # PD segmented (train/val/test)
    (seg_data_dir / "plantdoc").mkdir()
    for split in ["train", "val", "test"]:
        src = Path(f"/tmp/seg_plantdoc_{split}")
        if src.exists():
            (seg_data_dir / "plantdoc" / split).symlink_to(src.resolve())
    # PV segmented (use raw/segmented from Drive)
    pv_seg_raw = DATA_DIR / "plantvillage" / "raw" / "segmented"
    if not pv_seg_raw.exists():
        for alt in [DATA_DIR / "raw" / "segmented", Path("/content/folium_raw") / "plantvillage" / "segmented"]:
            if alt.exists():
                pv_seg_raw = alt
                break
    if pv_seg_raw.exists() and (pv_seg_raw / "train").exists():
        (seg_data_dir / "plantvillage").mkdir()
        for split in ["train", "val", "test"]:
            src = pv_seg_raw / split
            if src.exists():
                (seg_data_dir / "plantvillage" / split).symlink_to(src.resolve())
    # class_map.json
    import shutil
    shutil.copy(LOCAL_DATA_DIR / "class_map.json", seg_data_dir / "class_map.json")
    print(f"\nFull segmented data dir ready at {seg_data_dir}")
    print("PD classes:", sorted([d.name for d in (seg_data_dir / "plantdoc" / "test").iterdir()]))

---
## Step 10 - Backbone-lr + mixed-ratio sweep

**The highest-leverage experiment.** Two proven ingredients combined for the first time:
- Unfrozen backbone at lr=1e-5 (gave 0.66 field when unfrozen on PD-only)
- Mixed PV+PD training with tunable ratio (gave 0.95+ lab retention)

Nobody has tried them together. Run 3 short training sweeps:
- `--plantdoc-repeat 15` (~7% field share per epoch)
- `--plantdoc-repeat 20` (~9% field share)
- `--plantdoc-repeat 30` (~13% field share)

**Data:** If `USE_SEGMENTED=True` was set after Step 9, sweep uses segmented PV+PD.
If `USE_SEGMENTED=False` (default), sweep uses plain color images.

In [ ]:
# Decide: plain or segmented data?
# If Step 9 showed segmentation helps AND you ran Step 6b, set USE_SEGMENTED = True
USE_SEGMENTED = False  # Set to True manually if segmentation showed a jump

if USE_SEGMENTED:
    print("Using SEGMENTED data for sweep")
    SWEEP_DATA_DIR = Path("/tmp/seg_data")
    if not (SWEEP_DATA_DIR / "plantdoc" / "train").exists():
        print("ERROR: Segmented data not found. Run Step 6b first.")
else:
    print("Using PLAIN color data for sweep")
    SWEEP_DATA_DIR = LOCAL_DATA_DIR

print(f"Data dir: {SWEEP_DATA_DIR}")

In [ ]:
# Run 1: repeat=15 (RESUME from epoch 3 after quota disconnect, stop at epoch 5)
# NOTE: train.py has no LR scheduler (constant Adam), so --epochs 5 only
# controls where the loop stops - it cannot affect learning rates.
# Resume restores model + optimizer + scaler state (ml/train.py:414-420).
print("=" * 60)
print("10a) backbone-lr=1e-5, mix-with=plantdoc, repeat=15")
print("=" * 60)
cmd = [
    sys.executable, "-m", "ml.train",
    "--data-dir", str(SWEEP_DATA_DIR),
    "--dataset", "plantvillage",
    "--backbone", "resnet50",
    "--epochs", "5",
    "--lr", "1e-3",
    "--backbone-lr", "1e-5",
    "--unfreeze-blocks", "10",
    "--batch-size", "128",
    "--resume", str(CHECKPOINT_DIR / "plantvillage_s10_blr15_epoch03.pt"),
    "--augment",
    "--mix-with", "plantdoc",
    "--plantdoc-repeat", "15",
    "--tag", "s10_blr15",
    "--checkpoint-dir", str(CHECKPOINT_DIR),
]
result = run(cmd, cwd=str(REPO_DIR), label="sweep repeat=15 failed", stream=True)

In [ ]:
# Run 2: repeat=20 (capped at 5 epochs to fit Colab quota)
print("=" * 60)
print("10b) backbone-lr=1e-5, mix-with=plantdoc, repeat=20")
print("=" * 60)
cmd = [
    sys.executable, "-m", "ml.train",
    "--data-dir", str(SWEEP_DATA_DIR),
    "--dataset", "plantvillage",
    "--backbone", "resnet50",
    "--epochs", "5",
    "--lr", "1e-3",
    "--backbone-lr", "1e-5",
    "--unfreeze-blocks", "10",
    "--batch-size", "128",
    "--augment",
    "--mix-with", "plantdoc",
    "--plantdoc-repeat", "20",
    "--tag", "s10_blr20",
    "--checkpoint-dir", str(CHECKPOINT_DIR),
]
result = run(cmd, cwd=str(REPO_DIR), label="sweep repeat=20 failed", stream=True)

In [ ]:
# Run 3: repeat=30 (capped at 5 epochs to fit Colab quota)
print("=" * 60)
print("10c) backbone-lr=1e-5, mix-with=plantdoc, repeat=30")
print("=" * 60)
cmd = [
    sys.executable, "-m", "ml.train",
    "--data-dir", str(SWEEP_DATA_DIR),
    "--dataset", "plantvillage",
    "--backbone", "resnet50",
    "--epochs", "5",
    "--lr", "1e-3",
    "--backbone-lr", "1e-5",
    "--unfreeze-blocks", "10",
    "--batch-size", "128",
    "--augment",
    "--mix-with", "plantdoc",
    "--plantdoc-repeat", "30",
    "--tag", "s10_blr30",
    "--checkpoint-dir", str(CHECKPOINT_DIR),
]
result = run(cmd, cwd=str(REPO_DIR), label="sweep repeat=30 failed", stream=True)

## Step 11 - Evaluate all sweep checkpoints

In [ ]:
# Eval BOTH checkpoint flavors per tag:
# - best_*.pt  = best-val model (could legally be epoch03 if epochs 4/5
#   never beat its val acc) -> variant suffix _best
# - *_epoch05.pt = final-epoch model -> variant suffix _ep5
# Distinct variants land as distinct CSV rows (dedup keys on variant+checkpoint).
for tag, repeat in [("s10_blr15", 15), ("s10_blr20", 20), ("s10_blr30", 30)]:
    candidates = [
        (CHECKPOINT_DIR / f"best_plantvillage_{tag}.pt", f"{tag}_best"),
        (CHECKPOINT_DIR / f"plantvillage_{tag}_epoch05.pt", f"{tag}_ep5"),
    ]

    for ckpt, suffix in candidates:
        if not ckpt.exists():
            print(f"SKIP {suffix}: {ckpt.name} not found")
            continue

        print("=" * 60)
        print(f"11) Eval {suffix} (repeat={repeat})")
        print("=" * 60)

        # Eval on PlantVillage (lab)
        cmd = [
            sys.executable, "-m", "ml.evaluate",
            "--checkpoint", str(ckpt),
            "--data-dir", str(LOCAL_DATA_DIR),
            "--dataset", "plantvillage",
            "--split", "test",
            "--variant", f"{suffix}_lab",
            "--results", str(RESULTS_DIR / "ablation_results.csv"),
        ]
        result = run(cmd, cwd=str(REPO_DIR), label=f"eval {suffix} lab failed", stream=True)

        # Eval on PlantDoc (field)
        cmd = [
            sys.executable, "-m", "ml.evaluate",
            "--checkpoint", str(ckpt),
            "--data-dir", str(LOCAL_DATA_DIR),
            "--dataset", "plantdoc",
            "--split", "test",
            "--map-to-pv",
            "--variant", f"{suffix}_field",
            "--results", str(RESULTS_DIR / "ablation_results.csv"),
        ]
        result = run(cmd, cwd=str(REPO_DIR), label=f"eval {suffix} field failed", stream=True)

## Step 12 - Final verdict

**Decision table (decided before seeing numbers):**

| field F1 | lab F1 | Verdict | Next action |
|---|---|---|---|
| >=0.55 | >=0.90 | Strong pass | Run repeat=20 or 30 next session to confirm |
| >=0.55 | <0.90 | Field working, lab dipping | Run repeat=20/30 anyway (field is harder gate); flag, don't auto-pass |
| <0.45 | >=0.90 | Same shape as past failures | Don't jump to repeat=30. One repeat=20 for trend; no upward trend -> abandon/pivot |
| <0.45 | <0.90 | Both bad | Abandon sweep immediately, pivot to segmented path |

Read `_best` and `_ep5` rows per tag - if they differ a lot, epochs 4/5 changed things.

In [ ]:
df = pd.read_csv(str(RESULTS_DIR / "ablation_results.csv")).drop_duplicates()

def zone(field_f1, lab_f1):
    if field_f1 >= 0.55 and lab_f1 >= 0.90:
        return "STRONG PASS -> confirm with repeat=20/30"
    if field_f1 >= 0.55:
        return "field OK, lab dipping -> run 20/30 anyway, FLAGGED"
    if lab_f1 >= 0.90:
        return "lab-biased failure -> one repeat=20 for trend, else pivot"
    return "both bad -> ABANDON sweep, pivot to segmented path"

print("=== Sweep results ===")
print(f"{'variant':<28} {'lab F1':>8} {'field F1':>10}  verdict")
print("-" * 90)

for tag in ["s10_blr15", "s10_blr20", "s10_blr30"]:
    for suffix in [f"{tag}_best", f"{tag}_ep5"]:
        lab_row = df[df["variant"] == f"{suffix}_lab"]
        field_row = df[df["variant"] == f"{suffix}_field"]
        if len(lab_row) > 0 and len(field_row) > 0:
            lab_f1 = lab_row["f1"].iloc[0]
            field_f1 = field_row["f1"].iloc[0]
            print(f"{suffix:<28} {lab_f1:>8.4f} {field_f1:>10.4f}  {zone(field_f1, lab_f1)}")
        else:
            print(f"{suffix:<28} {'missing':>8} {'missing':>10}")

print("\nSegmentation results:")
seg_row = df[df["variant"] == "s10_seg_mixed_field"]
orig_row = df[df["variant"] == "s10_orig_mixed_field"]
if len(orig_row) > 0 and len(seg_row) > 0:
    print(f"  Original:  {orig_row['f1'].iloc[0]:.4f}")
    print(f"  Segmented: {seg_row['f1'].iloc[0]:.4f}")
    print(f"  Delta:     {seg_row['f1'].iloc[0] - orig_row['f1'].iloc[0]:+.4f}")